# Notebook 01 — Data Source Research

**AI Evolution Map · Data Pipeline**

---

## Purpose

This notebook documents the data sources selected for the AI Evolution Map project,
evaluates what each source provides, and establishes the research questions the
combined dataset can answer.

**Nothing is downloaded here.** The actual API calls happen in Notebook 02.
This notebook is the planning and reference document.

---

## Research Questions

The platform asks five core questions, each answerable with different data sources:

1. **Timeline**: What were the key milestones in AI development, and when did they occur?
2. **Publications**: How has AI research publication volume changed over time?
3. **Topics**: Which research topics dominate each era of AI development?
4. **Models**: How has the open-source AI model ecosystem evolved on Hugging Face?
5. **Coverage**: Which eras and subfields are most poorly represented in available data?

---

## Entity Types and Their Sources

| Entity | Primary Source | Secondary Source |
|---|---|---|
| Milestone | Hand-curated (with source citations) | Stanford AI Index |
| Concept | Hand-curated | arXiv categories |
| ResearchSource | arXiv API, OpenAlex API | Stanford AI Index CSVs |
| MetricObservation | Stanford AI Index CSVs | OpenAlex group-by counts |
| HF Model metadata | Hugging Face Hub API | — |


In [1]:
# Standard library and display setup
from IPython.display import display, Markdown
from datetime import datetime, timezone
import json

ACCESS_DATE = datetime.now(timezone.utc).strftime('%Y-%m-%d')
print(f'Notebook executed: {ACCESS_DATE}')

Notebook executed: 2026-10-23


## Source 1: Stanford AI Index

**URL**: https://aiindex.stanford.edu/report/
**Type**: Institutional Report / Dataset
**Access**: Static CSV/Excel file downloads (no API)

### What it provides
Annual curated report covering R&D trends, technical performance benchmarks, investment, 
workforce, and governance. Published once per year by Stanford HAI.

### Relevant for this project
- Annual publication counts by field (MetricObservation entities)
- Benchmark performance tables (anchors for Milestone entities)
- Investment and compute trends

### Fields available
Year, indicator, geography/country, value, category — varies by chapter CSV.

### Limitations
- Not primary data — Stanford aggregates from third parties (LinkedIn, McKinsey)
- Annual cadence; cannot track rapid model releases
- Some underlying sources are proprietary; only aggregated summaries are available
- Coverage starts as early as 2000 for some metrics but 2017+ for others

### Licensing
Free for academic use with attribution. Not CC-licensed. 
Do not redistribute underlying data commercially.

### Initial ingestion scope
**Start small**: Download one chapter CSV (e.g. the publications-over-time table) 
and validate the workflow before expanding.

## Source 2: OpenAlex API

**URL**: https://api.openalex.org/works
**Type**: Open Scholarly API
**Access**: REST API, no authentication required; email polite pool recommended

### What it provides
~300 million scholarly works with full metadata: titles, abstracts (inverted index),
authors, institutions, citation counts, topics, and DOIs.
Full CC0 license — completely free to use.

### Relevant for this project
- Publication counts per year (MetricObservation)
- Paper metadata → ResearchSource entities
- Topic distributions across eras
- Institution and country coverage for geographic analysis

### Key endpoint
```
GET https://api.openalex.org/works
    ?filter=primary_topic.field.id:11,publication_year:2010-2023
    &per-page=25
    &cursor=*
    &mailto=YOUR_EMAIL
```

### Rate limits
~100k credits/day for anonymous; polite pool (with mailto) gets higher priority.
Max 100 records per page. Cursor pagination for large result sets.

### Limitations
- Abstracts are inverted indexes, not plain text (requires reconstruction)
- Author disambiguation errors possible for common names
- Topic classification lag for emerging subfields

### Licensing
**CC0 1.0 Public Domain** — no restrictions.

## Source 3: arXiv API

**URL**: http://export.arxiv.org/api/query
**Type**: Preprint server API
**Access**: Public, no authentication. Response is Atom XML.

### What it provides
Metadata for 2.4M+ preprints. Categories cs.AI and cs.LG cover the AI/ML frontier.
Includes title, abstract (plain text!), authors, dates, categories, and DOI/journal refs.

### Relevant for this project
- ResearchSource entities for key AI papers
- Counting publications per year in cs.AI / cs.LG (MetricObservation)
- Linking models (HF) to foundational papers

### Key endpoint
```
GET http://export.arxiv.org/api/query
    ?search_query=cat:cs.AI
    &sortBy=submittedDate
    &sortOrder=descending
    &max_results=50
    &start=0
```

### Rate limits
**3 second minimum between requests** (hard requirement). No formal API key.
Maximum 2000 results per query; offset ceiling at 30,000.

### Limitations
- Preprints only — not peer-reviewed by arXiv
- Self-selection bias: CS/Physics/Math over-represented
- No citation data in the API
- 30k result cap per query; date-chunking needed for large sweeps

### Licensing
Metadata (title, abstract, authors): CC0 / public domain.
Full-text PDFs: author copyright. Use metadata only.

## Source 4: Hugging Face Hub API

**URL**: https://huggingface.co/api/models
**Type**: Model registry API
**Access**: Public; no auth for read-only metadata. Token recommended for higher limits.

### What it provides
Metadata for 1M+ public model repositories: model ID, task (pipeline_tag),
tags, downloads, likes, creation date, library, license.

### Relevant for this project
- Model ecosystem growth over time (MetricObservation: model counts per year)
- Connecting papers to models (arXiv IDs appear in model cards)
- Open-source AI adoption trends

### Key endpoint
```
GET https://huggingface.co/api/models
    ?sort=downloads
    &direction=-1
    &limit=50
    &pipeline_tag=text-generation
```

### Rate limits
No published hard limit for unauthenticated reads. Use ≥1.5s delay.
Pagination via HTTP Link header.

### Limitations
- Massive duplication: thousands of fine-tuned variants of a few base models
- Download counts skewed by CI/CD automation, not just human users
- Model card quality varies wildly; many have incomplete metadata
- Cannot enumerate all model lineage without parsing README text

### Licensing
Metadata collection for academic analysis is acceptable per Hugging Face ToS.
Per-model content licenses vary; always check individual license tags.

In [2]:
# Document the initial ingestion scope — what we plan to collect in Notebook 02

ingestion_plan = {
    'openalex': {
        'description': 'AI/ML works from OpenAlex, filtered by Machine Learning field (id:11)',
        'year_range': [2010, 2023],
        'max_records': 100,  # 4 pages × 25 records — expand after validation
        'purpose': 'ResearchSource entities + annual publication MetricObservations'
    },
    'arxiv': {
        'description': 'Recent cs.AI preprints',
        'query': 'cat:cs.AI',
        'max_results': 50,
        'purpose': 'ResearchSource entities for key AI papers'
    },
    'huggingface': {
        'description': 'Top text-generation models by downloads',
        'filter': 'pipeline_tag=text-generation',
        'limit': 50,
        'purpose': 'Model ecosystem growth analysis'
    },
    'stanford_ai_index': {
        'description': 'Manual download of the AI publications CSV',
        'url': 'https://github.com/AI-Index/AI-Index-Report',
        'purpose': 'MetricObservation entities for macro trends'
    }
}

print('=== Initial Ingestion Plan ===')
for source, details in ingestion_plan.items():
    print(f'\n[{source.upper()}]')
    for k, v in details.items():
        print(f'  {k}: {v}')

=== Initial Ingestion Plan ===

[OPENALEX]
  description: AI/ML works from OpenAlex, filtered by Machine Learning field (id:11)
  year_range: [2010, 2023]
  max_records: 100
  purpose: ResearchSource entities + annual publication MetricObservations

[ARXIV]
  description: Recent cs.AI preprints
  query: cat:cs.AI
  max_results: 50
  purpose: ResearchSource entities for key AI papers

[HUGGINGFACE]
  description: Top text-generation models by downloads
  filter: pipeline_tag=text-generation
  limit: 50
  purpose: Model ecosystem growth analysis

[STANFORD_AI_INDEX]
  description: Manual download of the AI publications CSV
  url: https://github.com/AI-Index/AI-Index-Report
  purpose: MetricObservation entities for macro trends


## Cross-Source Integration Plan

Records from different sources will be linked via shared keys:

| Link | Key | Notes |
|---|---|---|
| arXiv ↔ OpenAlex | DOI | arXiv publishes DOIs for accepted papers; OpenAlex indexes them |
| arXiv ↔ HF Hub | arXiv ID in model cards | Many HF models cite the paper they implement |
| OpenAlex ↔ Stanford | Year + topic category | Match publication year and broad topic area |
| HF Hub ↔ Timeline | Model name + creation date | Anchor a model release Milestone on the timeline |

**Important caveat**: Cross-source linkage is approximate for most fields 
(especially model name matching). We preserve uncertainty rather than assuming matches.

In [3]:
# Save the ingestion plan to data/reference/ so Notebook 02 can load it
import json
from pathlib import Path

ref_dir = Path('../data/reference')
ref_dir.mkdir(parents=True, exist_ok=True)

plan_path = ref_dir / 'ingestion_plan.json'
with open(plan_path, 'w') as f:
    json.dump({'accessed_at': ACCESS_DATE, 'sources': ingestion_plan}, f, indent=2)

print(f'Ingestion plan saved to: {plan_path}')

Ingestion plan saved to: ..\data\reference\ingestion_plan.json
